# iHydroCal Workflow 01 — Setup PEST Dataset

This notebook prepares a clean PEST/PEST++ dataset for SWAT+ / SWAT+ gwflow calibration.

**Main outputs** are created in `ihydrocal_workspace/main`, so the original `TxtInOut` folder stays clean.

Workflow:

1. Create/copy model-specific config files.
2. Optionally update `cha_ids` from `channels_gages.csv`.
3. Copy the model to `ihydrocal_workspace/main`.
4. Create `calibration.cal` and `calibration.cal.tpl`.
5. Extract selected channel output to `cha_flo_out_day.csv`.
6. Create `sim_stf_day.dat` and `sim_stf_day.dat.ins`.
7. Create the initial PEST++ control file.

## 0. Imports

In [ ]:
from pathlib import Path

from ihydrocal.core.config import init_project_config, print_config_summary
from ihydrocal.core.workspace import setup_workspace
from ihydrocal.core.mapping import update_ids_from_mapping
from ihydrocal.core.pest import create_pest_control_file

from ihydrocal.models.swatplus_gwflow.parameters import (
    get_active_parameters,
    write_calibration_cal,
    write_calibration_template,
)
from ihydrocal.models.swatplus_gwflow.outputs import extract_swatplus_channel_output_long
from ihydrocal.models.swatplus_gwflow.observations import prepare_streamflow_instruction_files

## 1. Define project paths

Update these two paths for your model. Forward slashes work on Windows and Linux.

In [ ]:
TXTINOUT_DIR = Path("C:/Users/seonggpa/Documents/projects/watersheds/Pecos/Analysis/calibration/TxtInOut_p_29hru_t10m")
CONFIG_FILE = Path("C:/Users/seonggpa/Documents/projects/watersheds/Pecos/Analysis/calibration/config/setup_swatplus.yml")

## 2. Create config files once

Run this cell only when starting a new project.

It copies `setup_swatplus.yml` and `swatp_pars_ihydrocal.db.csv` into the model-specific `config` folder.

After this, edit:

- `setup_swatplus.yml`
- `swatp_pars_ihydrocal.db.csv`

Do **not** rerun this with `overwrite=True` after editing, unless you want to replace your edits.

In [ ]:
# Uncomment only for first-time initialization.

# config_file = init_project_config(
#     txtinout_dir=TXTINOUT_DIR,
#     overwrite=False,
# )
# print(f"Created config: {config_file}")

## 3. Optional — update `cha_ids` from mapping file

Use this if you have many gages and a mapping file:

```text
config/channels_gages.csv
```

The file should contain a `channel_id` column. This cell updates the `cha_ids` list in `setup_swatplus.yml`.

In [ ]:
mapping_file = CONFIG_FILE.parent / "channels_gages.csv"

if mapping_file.exists():
    cha_ids = update_ids_from_mapping(
        config_file=CONFIG_FILE,
        mapping_file=mapping_file,
        id_col="channel_id",
        yaml_key="cha_ids",
    )
    print(f"Updated {len(cha_ids)} channel IDs in setup_swatplus.yml")
else:
    print(f"Mapping file not found; skipping cha_ids update: {mapping_file}")

## 4. Create workspace

This copies the original model into:

```text
ihydrocal_workspace/main
```

Generated files should be created in `main`, not in the original `TxtInOut`.

In [ ]:
cfg, workspace_dir, model_dir = setup_workspace(CONFIG_FILE)

print_config_summary(cfg)
print(f"Workspace directory : {workspace_dir}")
print(f"Main model directory: {model_dir}")

## 5. Create `calibration.cal` and template file

Rows with `flag = 1` in `swatp_pars_ihydrocal.db.csv` become active calibration parameters.

In [ ]:
parameter_db_name = cfg["input_files"]["swatplus"]["parameter_databases"][0]
parameter_db = cfg["config_dir"] / parameter_db_name

active = get_active_parameters(parameter_db)

cal_file = model_dir / "calibration.cal"
tpl_file = model_dir / "calibration.cal.tpl"

write_calibration_cal(active, cal_file)
write_calibration_template(active, cal_file, tpl_file)

print(f"Created calibration file: {cal_file}")
print(f"Created template file   : {tpl_file}")
print(f"Active parameters       : {len(active)}")

## 6. Extract selected channel output

This reads the large SWAT+ output file, for example `channel_sd_day.txt`, and extracts only selected channels and one variable such as `flo_out`.

The output is long-format:

```text
date, channel_id, simulated
```

Output file:

```text
main/cha_flo_out_day.csv
```

In [ ]:
channel_cfg = cfg["outputs"]["swatplus"]["channel"]
sim_channel_file = model_dir / "cha_flo_out_day.csv"

extract_swatplus_channel_output_long(
    output_file=model_dir / channel_cfg["file"],
    output_csv=sim_channel_file,
    value_col=channel_cfg["variables"][0],
    cha_ids=channel_cfg["cha_ids"],
    id_col=channel_cfg["id_col"],
)

print(f"Created extracted channel output: {sim_channel_file}")

## 7. Create streamflow DAT and instruction file

This combines:

- `stf_day.obd.csv`
- `channels_gages.csv`
- `cha_flo_out_day.csv`

and creates:

- `sim_stf_day.dat`
- `sim_stf_day.dat.ins`

Zero flows are kept. Blank and `-999` values are removed.

In [ ]:
output_dat, output_ins, obs_table = prepare_streamflow_instruction_files(
    site_col="SITENO",
    channel_col="channel_id",
    obs_file=cfg["config_dir"] / "stf_day.obd.csv",
    mapping_file=cfg["config_dir"] / "channels_gages.csv",
    sim_file=sim_channel_file,
    output_dat=model_dir / "sim_stf_day.dat",
    create_ins=True,
)

print(f"Created simulation DAT  : {output_dat}")
print(f"Created instruction file: {output_ins}")
print(f"Number of observations  : {len(obs_table)}")

## 8. Create initial PEST++ control file

This uses pyEMU to find `*.tpl` and `*.ins` files and create the initial PEST control file.

It also updates:

- actual observation values (`obsval`)
- observation groups (`obgnme`) by channel, such as `cha0015`
- parameter bounds, offsets, and groups from `swatp_pars_ihydrocal.db.csv`

The control file is written using `version=2`.

In [ ]:
pst_file = create_pest_control_file(
    cfg=cfg,
    model_dir=model_dir,
)

print(f"Created PEST control file: {pst_file}")

## 9. Setup complete

You should now have a clean PEST dataset inside:

```text
ihydrocal_workspace/main
```

Next, open `02_run_ies.ipynb` to run the initial PEST++ check, reweight, and start PESTPP-IES.